<a href="https://colab.research.google.com/github/natrask/ENM5320-2026/blob/main/old_material/VAE_DDPM_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Generative Modeling: VAE and Denoising Diffusion on MNIST

This notebook demonstrates two foundational generative models trained on MNIST:

**Part 1 — Variational Autoencoder (VAE)** following [Kingma & Welling (2014)](https://arxiv.org/abs/1312.6114). The core idea is to learn a latent representation $z$ of data $x$ by jointly training:
- An **encoder** $q_\phi(z|x)$ that maps images to a distribution in latent space
- A **decoder** $p_\theta(x|z)$ that reconstructs images from latent codes

Training maximizes the **Evidence Lower Bound (ELBO)**:
$$\mathcal{L}(\theta, \phi; x) = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - \mathrm{KL}(q_\phi(z|x) \| p(z))$$

The first term encourages faithful reconstruction; the second regularizes the latent space toward the prior $p(z) = \mathcal{N}(0, I)$.

**Part 2 — Denoising Diffusion Probabilistic Model (DDPM)** following [Ho et al. (2020)](https://arxiv.org/abs/2006.11239). Instead of learning a direct mapping from data to latent code, we:
- **Forward process**: gradually corrupt data by adding Gaussian noise over $T$ timesteps
- **Reverse process**: train a neural network $\epsilon_\theta(x_t, t)$ to predict the noise added at each step

Generation works by starting from pure noise $x_T \sim \mathcal{N}(0, I)$ and iteratively denoising.

**What you'll learn:**
1. How to implement a VAE with the reparameterization trick
2. How the ELBO loss balances reconstruction and regularization
3. How to walk through the latent space of a VAE
4. How to implement a DDPM with a simple U-Net denoiser
5. How to sample from a diffusion model

## 1. Setup and imports

All dependencies are standard PyTorch — no extra packages needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load MNIST

We load MNIST and normalize pixel values to $[0, 1]$. The VAE will model pixels as Bernoulli random variables (using binary cross-entropy loss), following the original paper.

In [ ]:
transform = transforms.ToTensor()  # scales to [0, 1]

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
print(f"Image shape: {train_dataset[0][0].shape}")

In [ ]:
# Visualize a few examples
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(train_dataset[i][0].squeeze(), cmap='gray')
    ax.set_title(str(train_dataset[i][1]), fontsize=9)
    ax.axis('off')
plt.suptitle('MNIST samples', fontsize=12)
plt.tight_layout()
plt.show()

---
# Part 1: Variational Autoencoder (VAE)

## 3. VAE architecture

Following Kingma & Welling, we use a simple MLP-based encoder and decoder. The encoder outputs the mean $\mu$ and log-variance $\log \sigma^2$ of the approximate posterior $q_\phi(z|x) = \mathcal{N}(\mu, \sigma^2 I)$.

The **reparameterization trick** allows us to backpropagate through the sampling step:
$$z = \mu + \sigma \odot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This moves the stochasticity into an input ($\epsilon$) rather than a parameter, making the gradient $\partial z / \partial \mu$ and $\partial z / \partial \sigma$ well-defined.

In [ ]:
class VAE(nn.Module):
    """Variational Autoencoder (Kingma & Welling, 2014).
    
    Architecture:
        Encoder: 784 -> 512 -> 256 -> (mu, logvar) of dim latent_dim
        Decoder: latent_dim -> 256 -> 512 -> 784
    """
    def __init__(self, latent_dim=20):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
        )
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Linear(512, 784),
            nn.Sigmoid(),  # output in [0, 1] for Bernoulli likelihood
        )
    
    def encode(self, x):
        """Map input to latent distribution parameters."""
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        """Sample z ~ N(mu, sigma^2 I) using the reparameterization trick."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        """Map latent code to reconstructed image."""
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

## 4. VAE loss function

The negative ELBO decomposes into two terms:

$$-\mathcal{L} = \underbrace{-\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{Reconstruction loss}} + \underbrace{\mathrm{KL}(q_\phi(z|x) \| p(z))}_{\text{KL regularization}}$$

For the **reconstruction loss**, with $p_\theta(x|z) = \prod_i \text{Bernoulli}(x_i; \hat{x}_i)$, this becomes the binary cross-entropy between input and reconstruction.

The **KL term** has a closed-form solution when both $q$ and $p(z)$ are Gaussian:
$$\mathrm{KL}(q \| p) = -\frac{1}{2}\sum_{j=1}^{d}\left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    """Compute the negative ELBO: reconstruction + KL divergence.
    
    Args:
        recon_x: reconstructed images, shape [B, 784]
        x: original images, shape [B, 1, 28, 28]
        mu: encoder mean, shape [B, latent_dim]
        logvar: encoder log-variance, shape [B, latent_dim]
    
    Returns:
        total loss, reconstruction loss, KL loss (all scalars, summed over batch)
    """
    # Reconstruction: binary cross-entropy (summed over pixels, averaged over batch)
    recon_loss = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum')
    
    # KL divergence: closed-form for Gaussian q vs N(0,I) prior
    # KL = -0.5 * sum(1 + log(sigma^2) - mu^2 - sigma^2)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + kl_loss, recon_loss, kl_loss

## 5. Train the VAE

We use Adam with the same learning rate ($10^{-3}$) as in the original paper. Training for 20 epochs on MNIST is sufficient to get good reconstructions and a smooth latent space.

In [ ]:
latent_dim = 20
vae = VAE(latent_dim=latent_dim).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

num_params = sum(p.numel() for p in vae.parameters())
print(f"VAE parameters: {num_params:,}")
print(f"Latent dimension: {latent_dim}")

In [ ]:
num_epochs = 20
vae_train_losses = []

for epoch in range(num_epochs):
    vae.train()
    epoch_loss = 0
    epoch_recon = 0
    epoch_kl = 0
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()
        
        recon, mu, logvar = vae(data)
        loss, recon_loss, kl_loss = vae_loss(recon, data, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_recon += recon_loss.item()
        epoch_kl += kl_loss.item()
    
    n = len(train_loader.dataset)
    vae_train_losses.append(epoch_loss / n)
    print(f"Epoch {epoch+1:2d}/{num_epochs}  "
          f"Loss: {epoch_loss/n:.2f}  "
          f"Recon: {epoch_recon/n:.2f}  "
          f"KL: {epoch_kl/n:.2f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(vae_train_losses, 'b-o', markersize=4)
plt.xlabel('Epoch'); plt.ylabel('Negative ELBO / sample')
plt.title('VAE Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 6. VAE results

### Reconstructions
The top row shows original test images; the bottom row shows the VAE's reconstructions. Because the latent bottleneck forces the model to compress to just 20 dimensions, reconstructions are slightly blurry — this is a well-known characteristic of VAEs.

In [ ]:
vae.eval()
with torch.no_grad():
    test_data = next(iter(test_loader))[0][:16].to(device)
    recon, _, _ = vae(test_data)
    recon = recon.view(-1, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(test_data[i].cpu().squeeze(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_title('Original', fontsize=9)
    
    axes[1, i].imshow(recon[i].cpu().squeeze(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_title('Reconstruction', fontsize=9)

plt.suptitle('VAE Reconstructions', fontsize=12)
plt.tight_layout()
plt.show()

### Sampling from the prior

Because the KL term regularizes $q_\phi(z|x)$ toward $\mathcal{N}(0, I)$, we can generate new images by sampling $z \sim \mathcal{N}(0, I)$ and decoding. This is the generative power of the VAE.

In [ ]:
with torch.no_grad():
    z_samples = torch.randn(16, latent_dim, device=device)
    generated = vae.decode(z_samples).view(-1, 1, 28, 28)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(generated[i].cpu().squeeze(), cmap='gray')
    ax.axis('off')
plt.suptitle('VAE: Samples from prior $z \sim \mathcal{N}(0, I)$', fontsize=12)
plt.tight_layout()
plt.show()

### Latent space interpolation

A well-structured latent space means that **linear interpolation** between two encoded images produces a smooth morphing in pixel space. We encode two test digits, interpolate their latent codes, and decode the intermediate points.

In [ ]:
vae.eval()
with torch.no_grad():
    # Pick two digits
    img1 = test_dataset[0][0].to(device)   # a '7'
    img2 = test_dataset[1][0].to(device)   # a '2'
    
    mu1, _ = vae.encode(img1.view(1, -1))
    mu2, _ = vae.encode(img2.view(1, -1))
    
    # Interpolate in latent space
    n_steps = 10
    alphas = torch.linspace(0, 1, n_steps, device=device)
    interpolations = []
    for alpha in alphas:
        z_interp = (1 - alpha) * mu1 + alpha * mu2
        img_interp = vae.decode(z_interp).view(28, 28)
        interpolations.append(img_interp.cpu())

fig, axes = plt.subplots(1, n_steps, figsize=(15, 2))
for i, img in enumerate(interpolations):
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'$\\alpha$={alphas[i]:.1f}', fontsize=8)
plt.suptitle('Latent space interpolation', fontsize=12)
plt.tight_layout()
plt.show()

### 2D latent space visualization

To visualize how the latent space is organized, we retrain a VAE with `latent_dim=2` and plot where each digit class lands. With a 2D latent space, we can also sweep a grid and decode every point to see the manifold of digits.

In [ ]:
# Train a 2D VAE (quick, small latent space)
vae2d = VAE(latent_dim=2).to(device)
optimizer2d = torch.optim.Adam(vae2d.parameters(), lr=1e-3)

for epoch in range(20):
    vae2d.train()
    for data, _ in train_loader:
        data = data.to(device)
        optimizer2d.zero_grad()
        recon, mu, logvar = vae2d(data)
        loss, _, _ = vae_loss(recon, data, mu, logvar)
        loss.backward()
        optimizer2d.step()
    if (epoch + 1) % 5 == 0:
        print(f"2D-VAE Epoch {epoch+1}/20  Loss: {loss.item()/len(data):.2f}")

In [ ]:
# Encode all test data and plot colored by digit class
vae2d.eval()
all_mu = []
all_labels = []
with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        mu, _ = vae2d.encode(data.view(-1, 784))
        all_mu.append(mu.cpu())
        all_labels.append(labels)

all_mu = torch.cat(all_mu, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Scatter plot of latent codes
scatter = ax1.scatter(all_mu[:, 0], all_mu[:, 1], c=all_labels, cmap='tab10',
                      s=2, alpha=0.5)
ax1.set_xlabel('$z_1$'); ax1.set_ylabel('$z_2$')
ax1.set_title('Latent space (colored by digit)')
plt.colorbar(scatter, ax=ax1, ticks=range(10))

# Decode a grid of latent points
n_grid = 20
grid_x = np.linspace(-3, 3, n_grid)
grid_y = np.linspace(-3, 3, n_grid)
canvas = np.zeros((28 * n_grid, 28 * n_grid))

with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z = torch.tensor([[xi, yi]], dtype=torch.float32, device=device)
            decoded = vae2d.decode(z).cpu().view(28, 28).numpy()
            canvas[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = decoded

ax2.imshow(canvas, cmap='gray', origin='lower',
           extent=[-3, 3, -3, 3])
ax2.set_xlabel('$z_1$'); ax2.set_ylabel('$z_2$')
ax2.set_title('Decoded latent grid')

plt.tight_layout()
plt.show()

---
# Part 2: Denoising Diffusion Probabilistic Model (DDPM)

## 7. Diffusion theory

The **forward process** adds Gaussian noise over $T$ timesteps:
$$q(x_t | x_{t-1}) = \mathcal{N}\!\left(x_t; \sqrt{1-\beta_t}\, x_{t-1},\; \beta_t I\right)$$

Thanks to the Gaussian chain rule, we can sample $x_t$ directly from $x_0$:
$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I)$$

where $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s=1}^t \alpha_s$.

The **reverse process** is parameterized as:
$$p_\theta(x_{t-1} | x_t) = \mathcal{N}\!\left(x_{t-1}; \mu_\theta(x_t, t),\; \sigma_t^2 I\right)$$

Following Ho et al., we train a network $\epsilon_\theta(x_t, t)$ to predict the noise $\epsilon$, and minimize:
$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\left[\|\epsilon - \epsilon_\theta(x_t, t)\|^2\right]$$

## 8. Noise schedule and diffusion utilities

In [ ]:
class DiffusionSchedule:
    """Linear noise schedule from Ho et al. (2020)."""
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.T = T
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_bar = torch.sqrt(self.alpha_bar)
        self.sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alpha_bar)
        self.sqrt_recip_alpha = torch.sqrt(1.0 / self.alphas)
        
        # For posterior q(x_{t-1} | x_t, x_0)
        alpha_bar_prev = F.pad(self.alpha_bar[:-1], (1, 0), value=1.0)
        self.posterior_var = self.betas * (1.0 - alpha_bar_prev) / (1.0 - self.alpha_bar)
    
    def q_sample(self, x0, t, noise=None):
        """Forward process: sample x_t from x_0 and t."""
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = self.sqrt_alpha_bar[t][:, None, None, None]
        sqrt_1mab = self.sqrt_one_minus_alpha_bar[t][:, None, None, None]
        return sqrt_ab * x0 + sqrt_1mab * noise

# Visualize the forward process
schedule = DiffusionSchedule(T=1000, device=device)

sample_img = train_dataset[0][0].unsqueeze(0).to(device)  # [1,1,28,28]
timesteps = [0, 50, 100, 250, 500, 750, 999]

fig, axes = plt.subplots(1, len(timesteps), figsize=(14, 2.5))
for i, t_val in enumerate(timesteps):
    t = torch.tensor([t_val], device=device)
    noisy = schedule.q_sample(sample_img, t)
    axes[i].imshow(noisy[0, 0].cpu(), cmap='gray')
    axes[i].set_title(f't={t_val}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Forward diffusion process: gradually adding noise', fontsize=12)
plt.tight_layout()
plt.show()

## 9. U-Net denoiser

We implement a lightweight U-Net with sinusoidal time embeddings. The network takes a noisy image $x_t$ and timestep $t$ as input and predicts the noise $\epsilon$ that was added.

This is intentionally kept small so it trains quickly on Colab. For higher quality results, you would scale up the channel counts and add attention layers.

In [ ]:
class SinusoidalPosEmb(nn.Module):
    """Sinusoidal positional embedding for timestep conditioning."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
    
    def forward(self, t):
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None].float() * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ResBlock(nn.Module):
    """Residual block with time embedding injection."""
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.GroupNorm(8, in_ch),
            nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_ch),
        )
        self.conv2 = nn.Sequential(
            nn.GroupNorm(8, out_ch),
            nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    
    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_mlp(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.skip(x)


class SimpleUNet(nn.Module):
    """Lightweight U-Net for MNIST denoising.
    
    Encoder: 1 -> 32 -> 64 (with 2x downsampling)
    Bottleneck: 64 -> 64
    Decoder: 128 -> 32 -> 1 (with 2x upsampling + skip connections)
    """
    def __init__(self, time_dim=128):
        super().__init__()
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        
        # Encoder
        self.enc1 = ResBlock(1, 32, time_dim)
        self.down1 = nn.Conv2d(32, 32, 4, stride=2, padding=1)   # 28->14
        self.enc2 = ResBlock(32, 64, time_dim)
        self.down2 = nn.Conv2d(64, 64, 4, stride=2, padding=1)   # 14->7
        
        # Bottleneck
        self.bottleneck = ResBlock(64, 64, time_dim)
        
        # Decoder
        self.up2 = nn.ConvTranspose2d(64, 64, 4, stride=2, padding=1)  # 7->14
        self.dec2 = ResBlock(128, 32, time_dim)  # 64 + 64 skip
        self.up1 = nn.ConvTranspose2d(32, 32, 4, stride=2, padding=1)  # 14->28
        self.dec1 = ResBlock(64, 32, time_dim)   # 32 + 32 skip
        
        # Output
        self.out = nn.Sequential(
            nn.GroupNorm(8, 32),
            nn.SiLU(),
            nn.Conv2d(32, 1, 1),
        )
    
    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        
        # Encoder
        h1 = self.enc1(x, t_emb)          # [B, 32, 28, 28]
        h2 = self.enc2(self.down1(h1), t_emb)  # [B, 64, 14, 14]
        
        # Bottleneck
        h = self.bottleneck(self.down2(h2), t_emb)  # [B, 64, 7, 7]
        
        # Decoder with skip connections
        h = self.up2(h)                     # [B, 64, 14, 14]
        h = self.dec2(torch.cat([h, h2], dim=1), t_emb)  # [B, 32, 14, 14]
        h = self.up1(h)                     # [B, 32, 28, 28]
        h = self.dec1(torch.cat([h, h1], dim=1), t_emb)  # [B, 32, 28, 28]
        
        return self.out(h)


unet = SimpleUNet().to(device)
num_params = sum(p.numel() for p in unet.parameters())
print(f"U-Net parameters: {num_params:,}")

## 10. Train the DDPM

Each training step:
1. Sample a batch of images $x_0$
2. Sample random timesteps $t \sim \text{Uniform}\{1, \ldots, T\}$
3. Sample noise $\epsilon \sim \mathcal{N}(0, I)$
4. Create noisy images $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\epsilon$
5. Predict $\hat\epsilon = \epsilon_\theta(x_t, t)$
6. Minimize $\|\epsilon - \hat\epsilon\|^2$

In [ ]:
ddpm_optimizer = torch.optim.Adam(unet.parameters(), lr=2e-4)
ddpm_epochs = 15
ddpm_losses = []

for epoch in range(ddpm_epochs):
    unet.train()
    epoch_loss = 0
    n_batches = 0
    
    for data, _ in train_loader:
        data = data.to(device)
        B = data.shape[0]
        
        # Sample random timesteps
        t = torch.randint(0, schedule.T, (B,), device=device)
        
        # Sample noise and create noisy images
        noise = torch.randn_like(data)
        x_t = schedule.q_sample(data, t, noise)
        
        # Predict noise
        noise_pred = unet(x_t, t)
        
        # Simple loss: MSE between true and predicted noise
        loss = F.mse_loss(noise_pred, noise)
        
        ddpm_optimizer.zero_grad()
        loss.backward()
        ddpm_optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    ddpm_losses.append(avg_loss)
    print(f"Epoch {epoch+1:2d}/{ddpm_epochs}  Loss: {avg_loss:.6f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(ddpm_losses, 'r-o', markersize=4)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('DDPM Training Loss')
plt.grid(True, alpha=0.3)
plt.show()

## 11. DDPM sampling

To generate images, we start from $x_T \sim \mathcal{N}(0, I)$ and iteratively apply the reverse process:
$$x_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t, t)\right) + \sigma_t z$$

where $z \sim \mathcal{N}(0, I)$ for $t > 1$ and $z = 0$ for $t = 1$.

In [ ]:
@torch.no_grad()
def ddpm_sample(model, schedule, n_samples=16, img_shape=(1, 28, 28), save_steps=None):
    """Generate samples via the reverse diffusion process."""
    model.eval()
    x = torch.randn(n_samples, *img_shape, device=device)
    intermediates = []
    
    for t_idx in reversed(range(schedule.T)):
        t = torch.full((n_samples,), t_idx, device=device, dtype=torch.long)
        
        # Predict noise
        eps_pred = model(x, t)
        
        # Compute mean of p(x_{t-1} | x_t)
        beta_t = schedule.betas[t_idx]
        sqrt_recip_alpha = schedule.sqrt_recip_alpha[t_idx]
        sqrt_1mab = schedule.sqrt_one_minus_alpha_bar[t_idx]
        
        mean = sqrt_recip_alpha * (x - beta_t / sqrt_1mab * eps_pred)
        
        if t_idx > 0:
            noise = torch.randn_like(x)
            sigma = torch.sqrt(schedule.posterior_var[t_idx])
            x = mean + sigma * noise
        else:
            x = mean
        
        if save_steps is not None and t_idx in save_steps:
            intermediates.append((t_idx, x.clone()))
    
    return x, intermediates

In [ ]:
# Generate samples
samples, _ = ddpm_sample(unet, schedule, n_samples=16)
samples = samples.clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    ax = axes[i // 8, i % 8]
    ax.imshow(samples[i, 0].cpu(), cmap='gray')
    ax.axis('off')
plt.suptitle('DDPM: Generated samples', fontsize=12)
plt.tight_layout()
plt.show()

### Visualize the reverse (denoising) process

We show snapshots of a single sample as it evolves from pure noise ($t = 999$) to a clean image ($t = 0$).

In [ ]:
# Generate one sample with intermediate steps saved
save_at = [999, 800, 600, 400, 200, 100, 50, 20, 10, 0]
_, intermediates = ddpm_sample(unet, schedule, n_samples=1, save_steps=save_at)
intermediates = sorted(intermediates, key=lambda x: -x[0])  # sort by t descending

fig, axes = plt.subplots(1, len(intermediates), figsize=(15, 2.5))
for i, (t_val, img) in enumerate(intermediates):
    axes[i].imshow(img[0, 0].clamp(0, 1).cpu(), cmap='gray')
    axes[i].set_title(f't={t_val}', fontsize=9)
    axes[i].axis('off')
plt.suptitle('Reverse diffusion: denoising from noise to image', fontsize=12)
plt.tight_layout()
plt.show()

---
## 12. Comparison: VAE vs DDPM

Let's compare the two models side by side.

In [ ]:
# VAE samples
vae.eval()
with torch.no_grad():
    z = torch.randn(8, latent_dim, device=device)
    vae_samples = vae.decode(z).view(-1, 1, 28, 28)

# DDPM samples
ddpm_samples, _ = ddpm_sample(unet, schedule, n_samples=8)
ddpm_samples = ddpm_samples.clamp(0, 1)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(8):
    axes[0, i].imshow(vae_samples[i, 0].cpu(), cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('VAE', fontsize=10)
    
    axes[1, i].imshow(ddpm_samples[i, 0].cpu(), cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('DDPM', fontsize=10)

plt.suptitle('VAE vs DDPM: Generated MNIST digits', fontsize=12)
plt.tight_layout()
plt.show()

**Observations:**
- The **VAE** generates images that are recognizable but tend to be **blurry**. This is a consequence of the Gaussian decoder assumption and the amortized variational inference.
- The **DDPM** produces **sharper** images by iteratively refining through many denoising steps.
- The VAE generates images in a **single forward pass** (fast), while the DDPM requires **1000 sequential denoising steps** (slow).
- The VAE provides a **structured latent space** (useful for interpolation, clustering), while the DDPM does not have an explicit latent representation.

## 13. Exercises

1. **$\beta$-VAE**: Add a weight $\beta$ to the KL term: $\mathcal{L} = \text{Recon} + \beta \cdot \text{KL}$. Try $\beta = 0.1, 1, 10$. How does this affect sample quality vs. latent space structure?
2. **Convolutional VAE**: Replace the MLP encoder/decoder with convolutional layers. Does this improve reconstruction quality?
3. **Fewer diffusion steps**: Reduce $T$ from 1000 to 100 or 50. How does sample quality degrade? Investigate DDIM-style sampling (deterministic) as an alternative.
4. **Conditional generation**: Modify the DDPM to accept class labels as additional input (class-conditional diffusion). Can you generate specific digits on command?
5. **FID score**: Implement or use a library to compute the Fréchet Inception Distance between generated samples and real test data for both models. Which scores better?
6. **Longer training**: The DDPM especially benefits from more epochs. Train for 50+ epochs and compare.